Notebook taken from [this repo](https://github.com/enescingoz/colab-llm/tree/main?tab=readme-ov-file). Readme contains useful information.

[Reddit thread](https://www.reddit.com/r/LocalLLaMA/comments/1jz2am6/run_local_llms_in_google_colab_for_free_with_gpu/) for discussions.

In [ ]:
!nvidia-smi

Wed Jan 28 17:56:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Model selection
MODEL_NAMES = [
    "maryasov/qwen2.5-coder-cline:7b-instruct-q8_0",
    "gpt-oss",
]
%env OLLAMA_CONTEXT_LENGTH=16384
%env OLLAMA_HOST=0.0.0.0
%env OLLAMA_KEEP_ALIVE=-1


env: OLLAMA_CONTEXT_LENGTH=16384
env: OLLAMA_HOST=0.0.0.0
env: OLLAMA_KEEP_ALIVE=-1


In [4]:
!apt-get install -y lshw pciutils
!nvcc --version
!nvidia-smi

from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"\n🧠 Available RAM: {ram_gb:.1f} GB")
print("✅ High-RAM runtime!" if ram_gb >= 20 else "❌ Not a high-RAM runtime.")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids
0 upgraded, 5 newly installed, 0 to remove and 41 not upgraded.
Need to get 883 kB of archives.
After this operation, 3,256 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2build1 [321 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Fetched 883 kB in 2s (588 kB/s)  
Selecting previously unselected package 

In [5]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (497 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121722 files and directories currently i

In [6]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import subprocess
import time
import requests
import threading

# Start ollama serve in a background thread
def start_ollama():
    subprocess.call(['ollama', 'serve'])

ollama_thread = threading.Thread(target=start_ollama)
ollama_thread.daemon = True
ollama_thread.start()

# Pull model (this also verifies Ollama CLI is ready)
#!ollama pull {MODEL_NAMES}

# Wait for Ollama HTTP API to be ready
def wait_for_ollama(timeout=60):
    for i in range(timeout):
        try:
            r = requests.get("http://localhost:11434")
            if r.status_code in [200, 404]:
                print(f"✅ Ollama is up (after {i+1}s).")
                return
        except requests.exceptions.ConnectionError:
            pass
        print(f"⏳ Waiting for Ollama to start... {i+1}s")
        time.sleep(1)
    raise RuntimeError("❌ Ollama did not start in time.")

wait_for_ollama()


⏳ Waiting for Ollama to start... 1s
✅ Ollama is up (after 2s).


In [8]:
!ollama pull gpt-oss

In [9]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


--2026-01-28 18:00:22--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.1.2/cloudflared-linux-amd64 [following]
--2026-01-28 18:00:22--  https://github.com/cloudflare/cloudflared/releases/download/2026.1.2/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/31ade8d6-2fcd-4925-9218-5534d27a01dc?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-28T18%3A40%3A32Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-28T1

In [10]:
import re

# Run cloudflared tunnel in background and get the public URL
cloudflared_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    match = re.search(r'(https://.*\.trycloudflare\.com)', line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\n✅ Public URL for Ollama:\n{public_url}")
else:
    raise RuntimeError("❌ Could not find public Cloudflare URL.")


2026-01-28T18:00:23Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-01-28T18:00:23Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-01-28T18:00:27Z INF +--------------------------------------------------------------------------------------------+
2026-01-28T18:00:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-01-28T18:00:27Z INF |  https://answer-encouraging-asian-two.trycloudflare.co

In [17]:
import json

data = {
    "model": "gpt-oss",
    "prompt": "Why is the moon yellow?",
    "stream": False
}

response = requests.post(f"{public_url}/api/generate", json=data)
print(response.json())


{'model': 'gpt-oss', 'created_at': '2026-01-28T18:11:05.852604463Z', 'response': '**Short answer**\n\nThe Moon itself isn’t yellow at all – it reflects the sunlight that hits it.  \nWhen that reflected light has to travel through the Earth’s atmosphere before it reaches our eyes, the blue part of the spectrum is scattered away (Rayleigh scattering) and we see a yellow‑to‑orange tint. The effect is strongest when the Moon is low on the horizon, where its light has to cross the greatest amount of air.\n\n---\n\n## 1. The Moon does not produce light of its own\n\n- **Reflection, not emission.**  \n  The Moon’s surface reflects about 12\u202f% of the sunlight that strikes it (its albedo).  \n- **Spectrum of sunlight.**  \n  Sunlight arriving at the Moon is essentially “white” (a mix of all visible wavelengths). So, in the absence of any other effect, the Moon would look white or very pale gray.\n\n## 2. The Earth’s atmosphere is the “tinting” agent\n\n### Rayleigh scattering\n\n- **How it 

In [16]:
data = {
    "model": "gpt-oss",
    "system": "you are an information extraction system which always outputs JSON, using the following fields: First Name, Last Name, Age, Sex",
    "prompt": "Hello! My name is Niccolò Ananas and I'm 27M. ",
    "stream": False
}

response = requests.post(f"{public_url}/api/generate", json=data)
print(response.json())

{'model': 'gpt-oss', 'created_at': '2026-01-28T18:10:23.721282403Z', 'response': '{\n  "First Name": "Niccolò",\n  "Last Name": "Ananas",\n  "Age": 27,\n  "Sex": "M"\n}', 'thinking': 'We need to output JSON with fields: First Name, Last Name, Age, Sex. Input: "Hello! My name is Niccolò Ananas and I\'m 27M."\n\nWe parse: First Name: Niccolò; Last Name: Ananas; Age: 27; Sex: M.\n\nWe need to produce JSON. Ensure proper keys exactly as specified. Use double quotes. Should be like:\n\n{\n  "First Name": "Niccolò",\n  "Last Name": "Ananas",\n  "Age": 27,\n  "Sex": "M"\n}\n\nCheck if age numeric, sex string. Yes. Ensure no extra.', 'done': True, 'done_reason': 'stop', 'context': [200006, 17360, 200008, 3575, 553, 17554, 162016, 11, 261, 4410, 6439, 2359, 22203, 656, 7788, 17527, 558, 87447, 100594, 25, 220, 1323, 19, 12, 3218, 198, 6576, 3521, 25, 220, 1323, 21, 12, 2290, 12, 2029, 279, 30377, 289, 25, 14093, 279, 2, 13888, 18403, 25, 8450, 11, 49159, 11, 1721, 13, 21030, 2804, 413, 7360, 39